# Build EarthCatalog from Scratch

This notebook provisions a Coiled cluster and runs a full catalog build against the ITS_LIVE S3 Inventory.

**Steps:**
1. Set all parameters in the `Parameters` cell below
2. Verify AWS credentials
3. Provision a Coiled cluster (or paste an existing scheduler address)
4. Run `run_backfill.py` — phases: Inventory scan → STAC fetch → Compact → Register
5. Inspect the resulting catalog
6. Shut down the cluster

> ⚠️ **Check `CATALOG_BASE` before running.**  The default value points to a scratch
> prefix that is distinct from the live catalog at `test-space/stac/catalog`.  A safety
> guard cell will stop you if they match.

## Parameters

In [ ]:
# ============================================================
# DESTINATION  — edit before running
# ============================================================

# All outputs (warehouse, indexes, earthcatalog.db) land under this prefix.
# Must be different from "s3://its-live-data/test-space/stac/catalog".
CATALOG_BASE   = "s3://its-live-data/test-space/stac/scratch-build-01"

# Derived — no need to edit unless you want a non-standard layout
WAREHOUSE      = f"{CATALOG_BASE.rstrip('/')}/warehouse"
STAGING        = f"{CATALOG_BASE.rstrip('/')}/ingest"
HASH_INDEX     = f"{CATALOG_BASE.rstrip('/')}/warehouse_id_hashes.parquet"
LOCAL_CATALOG  = "/tmp/earthcatalog_scratch.db"   # SQLite path on this machine

# Catalog key within the bucket (used by store_config for upload)
_base_key      = CATALOG_BASE.removeprefix("s3://").split("/", 1)[1].rstrip("/")
CATALOG_KEY    = f"{_base_key}/earthcatalog.db"
LOCK_KEY       = f"{_base_key}/.lock"

# ============================================================
# INVENTORY
# ============================================================

INVENTORY_MANIFEST = (
    "s3://pds-buckets-its-live-logbucket-70tr3aw5f2op/inventory/"
    "velocity_image_pair/its-live-data/VelocityGranuleInventory/"
    "2026-08-03T01-00Z/manifest.json"
)

# ============================================================
# COILED CLUSTER
# ============================================================

# Paste a scheduler address here to reuse a running cluster and skip
# provisioning.  Example: "tls://scheduler-abc123.us-west-2.aws.dask.host:8786"
# Leave as None to provision a fresh cluster.
COILED_SCHEDULER_ADDRESS = None

COILED_N_WORKERS          = 20
COILED_VM_TYPE            = "c6i.2xlarge"        # 8 vCPU, 16 GB RAM
COILED_THREADS_PER_WORKER = 2
COILED_CLUSTER_NAME       = "earthcatalog-scratch-build"
COILED_REGION             = "us-west-2"
COILED_SPOT_POLICY        = "spot_with_fallback" # "spot", "on_demand"

# ============================================================
# INGEST TUNING
# ============================================================

CHUNK_SIZE        = 100_000  # inventory items per chunk Parquet (Phase 1)
COMPACT_ROWS      = 100_000  # max rows per output GeoParquet file (Phase 3)
FETCH_CONCURRENCY = 256      # async STAC-JSON fetch concurrency per worker
H3_RESOLUTION     = 3        # H3 grid resolution (3 = ~830 km² cells)

# Set to an integer to build only the first N items (for smoke-testing)
LIMIT             = None

# ============================================================
# AWS
# ============================================================

AWS_REGION = "us-west-2"

# --- summary ---
print("Parameters")
print(f"  Catalog base  : {CATALOG_BASE}")
print(f"  Warehouse     : {WAREHOUSE}")
print(f"  Staging       : {STAGING}")
print(f"  Hash index    : {HASH_INDEX}")
print(f"  Catalog key   : {CATALOG_KEY}")
print(f"  Local DB      : {LOCAL_CATALOG}")
print(f"  Inventory     : {INVENTORY_MANIFEST}")
print(f"  Workers       : {COILED_N_WORKERS}× {COILED_VM_TYPE}")
print(f"  Chunk size    : {CHUNK_SIZE:,}")
print(f"  Limit         : {LIMIT}")

## Safety check

In [ ]:
PROTECTED = "s3://its-live-data/test-space/stac/catalog"

if CATALOG_BASE.rstrip("/") == PROTECTED.rstrip("/"):
    raise ValueError(
        f"CATALOG_BASE is set to the production path ({PROTECTED}).\n"
        "Change it to a new prefix before proceeding."
    )

print(f"OK — {CATALOG_BASE!r} does not clobber {PROTECTED!r}")

## Verify AWS credentials

In [ ]:
import boto3

identity = boto3.client("sts", region_name=AWS_REGION).get_caller_identity()
print(f"Account : {identity['Account']}")
print(f"UserId  : {identity['UserId']}")
print(f"ARN     : {identity['Arn']}")

## Provision Coiled cluster

In [ ]:
import glob
import os
import subprocess
import sys
import tempfile

cluster = None
client  = None

if COILED_SCHEDULER_ADDRESS:
    from dask.distributed import Client
    print(f"Connecting to: {COILED_SCHEDULER_ADDRESS}")
    client = Client(COILED_SCHEDULER_ADDRESS)
    scheduler_address = COILED_SCHEDULER_ADDRESS
else:
    import coiled
    from dask.distributed import Client

    # Build a wheel from the local repo so workers run the current dev version
    wheel_dir = tempfile.mkdtemp(prefix="earthcatalog-wheel-")
    repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    print(f"Building wheel from {repo_root} …")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "wheel", ".", "--no-deps", "-w", wheel_dir, "--quiet"],
        cwd=repo_root,
    )
    wheels = glob.glob(f"{wheel_dir}/*.whl")
    assert wheels, f"No .whl found in {wheel_dir}"
    wheel_path = wheels[0]
    print(f"Built: {os.path.basename(wheel_path)}")

    with open(wheel_path, "rb") as fh:
        whl_bytes = fh.read()
    whl_name = os.path.basename(wheel_path)

    def _install_wheel(whl_bytes, whl_name):
        import os, subprocess, sys, tempfile
        subprocess.check_call(
            [sys.executable, "-m", "pip", "uninstall", "-y", "earthcatalog", "--quiet"],
        )
        td = tempfile.mkdtemp()
        path = os.path.join(td, whl_name)
        with open(path, "wb") as f:
            f.write(whl_bytes)
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--no-deps", path, "--quiet"],
        )
        for mod in list(sys.modules):
            if "earthcatalog" in mod:
                del sys.modules[mod]

    print(f"Provisioning '{COILED_CLUSTER_NAME}' — {COILED_N_WORKERS}× {COILED_VM_TYPE} ({COILED_SPOT_POLICY}) …")
    cluster = coiled.Cluster(
        n_workers=COILED_N_WORKERS,
        worker_vm_types=[COILED_VM_TYPE],
        region=COILED_REGION,
        name=COILED_CLUSTER_NAME,
        worker_options={"nthreads": COILED_THREADS_PER_WORKER},
        spot_policy=COILED_SPOT_POLICY,
    )
    client = Client(cluster)
    scheduler_address = client.scheduler.address

    print(f"Dashboard        : {client.dashboard_link}")
    print(f"Scheduler address: {scheduler_address}")
    print(f"  (save this to COILED_SCHEDULER_ADDRESS to reuse the cluster)")

    print(f"Installing wheel on {COILED_N_WORKERS} workers ({len(whl_bytes):,} bytes) …")
    client.run(_install_wheel, whl_bytes=whl_bytes, whl_name=whl_name)
    print("Workers ready.")

## Run backfill

Calls `scripts/run_backfill.py` as a subprocess with `--coiled-scheduler-address` so the
cluster provisioned above is reused — no new cluster is created.  All store construction
and credential handling is done by the script exactly as it is in CI.

| Phase | What happens |
|-------|--------------|
| 1 — Inventory scan | Reads S3 Inventory Parquets, writes key-chunks to staging |
| 2 — STAC fetch | Workers fetch STAC JSON per key, write GeoParquet shards to staging |
| 3 — Compact | Merges shards into per-(cell, year) partition files in warehouse |
| 4 — Register | Registers files in Iceberg, uploads `earthcatalog.db` to S3 |

Expected runtime on 20× c6i.2xlarge: **~4–6 hours** for the full ~45 M-item catalog.

In [ ]:
import subprocess, sys, os

repo_root  = os.path.abspath(os.path.join(os.getcwd(), ".."))
script     = os.path.join(repo_root, "scripts", "run_backfill.py")

cmd = [
    sys.executable, script,
    "--inventory",              INVENTORY_MANIFEST,
    "--catalog",                LOCAL_CATALOG,
    "--warehouse",              WAREHOUSE,
    "--staging",                STAGING,
    "--hash-index",             HASH_INDEX,
    "--update-hash-index",
    "--update-source-index",
    "--chunk-size",             str(CHUNK_SIZE),
    "--compact-rows",           str(COMPACT_ROWS),
    "--fetch-concurrency",      str(FETCH_CONCURRENCY),
    "--coiled-scheduler-address", scheduler_address,
]

if H3_RESOLUTION is not None:
    cmd += ["--h3-resolution", str(H3_RESOLUTION)]

if LIMIT is not None:
    cmd += ["--limit", str(LIMIT)]

# Override store_config catalog key to point at the scratch prefix
# (run_backfill.py hard-codes the production key; patch via env var)
env = os.environ.copy()
env["EARTHCATALOG_CATALOG_KEY"] = CATALOG_KEY
env["EARTHCATALOG_LOCK_KEY"]    = LOCK_KEY

print("Running:")
print("  " + "\n  ".join(cmd))
print()

# Stream output live
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, env=env, cwd=repo_root)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"run_backfill.py exited with code {proc.returncode}")

## Inspect the new catalog

In [ ]:
import subprocess, sys, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

result = subprocess.run(
    [
        sys.executable, os.path.join(repo_root, "scripts", "info.py"),
        "--catalog",    LOCAL_CATALOG,
        "--warehouse",  WAREHOUSE,
        "--hash-index", HASH_INDEX,
    ],
    capture_output=True,
    text=True,
    cwd=repo_root,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## Shut down the cluster

Run this cell when the build is complete.  Coiled also enforces an idle timeout,
but explicit shutdown avoids unnecessary charges.

In [ ]:
if client:
    client.close()
if cluster:
    cluster.close()
    print("Cluster shut down.")
else:
    print("Using pre-existing scheduler — nothing to shut down from this notebook.")